In [151]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [152]:
nri_2023 = pd.read_csv("../../01_original_data/NRI/2023/NRI_Table_Counties.csv")
nri_2021 = pd.read_csv("../../01_original_data/NRI/2021/NRI_Table_Counties.csv")
nri_2020 = pd.read_csv("../../01_original_data/NRI/2020/NRI_Table_Counties.csv")

nri = pd.concat([nri_2023, nri_2021, nri_2020], ignore_index=True)
nri.columns = [re.sub(r"\s+", "_", c.strip().lower()) for c in nri.columns]
nri["stcofips"] = nri["statefips"].astype(str).str.zfill(2) + nri["countyfips"].astype(
    str
).str.zfill(3)
nri = nri[
    [
        "state",
        "stateabbrv",
        "county",
        "statefips",
        "stcofips",
        "population",
        "risk_value",
        "risk_score",
        "risk_ratng",
        "nri_ver",
    ]
]

nri["nri_ver"] = (
    nri["nri_ver"]
    .map({"October 2020": 2020, "November 2021": 2021, "March 2023": 2023})
    .astype(int)
)

# nri.info()
print(len(nri))
nri.head(10)

9515


,state,stateabbrv,county,statefips,stcofips,population,risk_value,risk_score,risk_ratng,nri_ver
0,Alabama,AL,Autauga,1,01001,58764,6.156054e+06,49.220490,Relatively Low,2023
1,Alabama,AL,Baldwin,1,01003,231365,2.106327e+08,97.709195,Relatively High,2023
2,Alabama,AL,Barbour,1,01005,25160,7.412840e+06,56.188355,Relatively Low,2023
3,Alabama,AL,Bibb,1,01007,22239,3.863747e+06,32.484887,Very Low,2023
4,Alabama,AL,Blount,1,01009,58992,1.023854e+07,65.128858,Relatively Low,2023
5,Alabama,AL,Bullock,1,01011,10326,3.861868e+06,32.453070,Very Low,2023
6,Alabama,AL,Butler,1,01013,19015,6.978411e+06,54.056634,Relatively Low,2023
7,Alabama,AL,Calhoun,1,01015,116250,2.623382e+07,84.377983,Relatively Moderate,2023
8,Alabama,AL,Chambers,1,01017,34738,5.041150e+06,42.348075,Very Low,2023
9,Alabama,AL,Cherokee,1,01019,24933,5.005164e+06,42.061724,Very Low,2023


In [153]:
hpi = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/hpi_master.csv"
)
# print(min(hpi["yr"]))
hpi = hpi[
    (hpi["level"] == "MSA") & (hpi["yr"].isin([2019, 2020, 2021, 2022, 2023]))
]  # only keep data for msa level
hpi = hpi[["place_name", "place_id", "yr", "period", "index_nsa"]]
# hpi.info()
# hpi.head(10)

In [154]:
hpi = hpi.groupby(["place_id", "yr"], as_index=False).agg(
    {
        "place_name": "first",
        "place_id": "first",
        "yr": "first",
        "index_nsa": "mean",
    }
)

hpi["index_prev"] = hpi.groupby(["place_id"])["index_nsa"].shift(1)
hpi = hpi[hpi["yr"].isin([2020, 2021, 2022, 2023])]
hpi["hpi_change"] = hpi["index_nsa"] - hpi["index_prev"]
hpi

,place_name,place_id,yr,index_nsa,index_prev,hpi_change
1,"Abilene, TX",10180,2020,235.9400,225.7875,10.1525
2,"Abilene, TX",10180,2021,266.3675,235.9400,30.4275
3,"Abilene, TX",10180,2022,300.3175,266.3675,33.9500
4,"Abilene, TX",10180,2023,327.3250,300.3175,27.0075
6,"Akron, OH",10420,2020,170.0325,160.8900,9.1425
...,...,...,...,...,...,...
2044,"Yuba City, CA",49700,2023,359.3725,362.1900,-2.8175
2046,"Yuma, AZ",49740,2020,207.5775,193.8850,13.6925
2047,"Yuma, AZ",49740,2021,249.3650,207.5775,41.7875
2048,"Yuma, AZ",49740,2022,302.8575,249.3650,53.4925


In [155]:
xwalk = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/list1_2023.csv"
)
xwalk.columns = (
    xwalk.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
)
xwalk["county_fips5"] = (
    xwalk["fips_state_code"].astype(str).str.upper().str.strip().str.zfill(2)
    + xwalk["fips_county_code"].astype(str).str.zfill(3).str.strip()
)
xwalk = xwalk[
    [
        "cbsa_code",
        "cbsa_title",
        "county_fips5",
        "county/county_equivalent",
        "state_name",
    ]
]
xwalk["cbsa_code"] = xwalk["cbsa_code"].astype(str)
# xwalk.info()
print(len(xwalk))
xwalk.head(10)

1915


,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name
0,10100,"Aberdeen, SD",46013,Brown County,South Dakota
1,10100,"Aberdeen, SD",46045,Edmunds County,South Dakota
2,10140,"Aberdeen, WA",53027,Grays Harbor County,Washington
3,10180,"Abilene, TX",48059,Callahan County,Texas
4,10180,"Abilene, TX",48253,Jones County,Texas
5,10180,"Abilene, TX",48441,Taylor County,Texas
6,10220,"Ada, OK",40123,Pontotoc County,Oklahoma
7,10300,"Adrian, MI",26091,Lenawee County,Michigan
8,10380,"Aguadilla, PR",72003,Aguada Municipio,Puerto Rico
9,10380,"Aguadilla, PR",72005,Aguadilla Municipio,Puerto Rico


In [156]:
# merge NRI with crosswalk on county fips
nri = nri.merge(
    xwalk[["cbsa_code", "county_fips5"]],
    left_on="stcofips",
    right_on="county_fips5",
    how="left",
).dropna(subset="cbsa_code")
print(len(nri))
nri.head(10)

5576


,state,stateabbrv,county,statefips,stcofips,population,risk_value,risk_score,risk_ratng,nri_ver,cbsa_code,county_fips5
0,Alabama,AL,Autauga,1,01001,58764,6.156054e+06,49.220490,Relatively Low,2023,33860,01001
1,Alabama,AL,Baldwin,1,01003,231365,2.106327e+08,97.709195,Relatively High,2023,19300,01003
2,Alabama,AL,Barbour,1,01005,25160,7.412840e+06,56.188355,Relatively Low,2023,21640,01005
3,Alabama,AL,Bibb,1,01007,22239,3.863747e+06,32.484887,Very Low,2023,13820,01007
4,Alabama,AL,Blount,1,01009,58992,1.023854e+07,65.128858,Relatively Low,2023,13820,01009
7,Alabama,AL,Calhoun,1,01015,116250,2.623382e+07,84.377983,Relatively Moderate,2023,11500,01015
8,Alabama,AL,Chambers,1,01017,34738,5.041150e+06,42.348075,Very Low,2023,29300,01017
10,Alabama,AL,Chilton,1,01021,44999,8.034569e+06,58.542794,Relatively Low,2023,13820,01021
15,Alabama,AL,Coffee,1,01031,53391,4.035483e+07,89.086860,Relatively Moderate,2023,21460,01031
16,Alabama,AL,Colbert,1,01033,57133,1.303910e+07,71.905822,Relatively Low,2023,22520,01033


In [157]:
# merge NRI with HPI on msa code
# print(hpi["yr"].value_counts())

nri = nri.merge(
    hpi,
    left_on=["cbsa_code", "nri_ver"],
    right_on=["place_id", "yr"],
    how="left",
).dropna(subset="index_nsa")

nri["yr"] = nri["yr"].astype(int)

print(nri.isna().sum())
print(len(nri))
nri.head(10)
# nri["yr"].value_counts()
# nri[nri["yr"].isna()]

state              0
stateabbrv         0
county             0
statefips          0
stcofips           0
population         0
risk_value      2078
risk_score         0
risk_ratng         0
nri_ver            0
cbsa_code          0
county_fips5       0
place_name         0
place_id           0
yr                 0
index_nsa          0
index_prev         0
hpi_change         0
dtype: int64
3117


,state,stateabbrv,county,statefips,stcofips,population,risk_value,risk_score,risk_ratng,nri_ver,cbsa_code,county_fips5,place_name,place_id,yr,index_nsa,index_prev,hpi_change
0,Alabama,AL,Autauga,1,01001,58764,6.156054e+06,49.220490,Relatively Low,2023,33860,01001,"Montgomery, AL",33860,2023,205.86250,195.3500,10.51250
1,Alabama,AL,Baldwin,1,01003,231365,2.106327e+08,97.709195,Relatively High,2023,19300,01003,"Daphne-Fairhope-Foley, AL",19300,2023,363.14250,333.1125,30.03000
3,Alabama,AL,Bibb,1,01007,22239,3.863747e+06,32.484887,Very Low,2023,13820,01007,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
4,Alabama,AL,Blount,1,01009,58992,1.023854e+07,65.128858,Relatively Low,2023,13820,01009,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
5,Alabama,AL,Calhoun,1,01015,116250,2.623382e+07,84.377983,Relatively Moderate,2023,11500,01015,"Anniston-Oxford, AL",11500,2023,261.24750,243.6425,17.60500
7,Alabama,AL,Chilton,1,01021,44999,8.034569e+06,58.542794,Relatively Low,2023,13820,01021,"Birmingham, AL",13820,2023,324.80125,308.8250,15.97625
9,Alabama,AL,Colbert,1,01033,57133,1.303910e+07,71.905822,Relatively Low,2023,22520,01033,"Florence-Muscle Shoals, AL",22520,2023,271.35750,250.2950,21.06250
15,Alabama,AL,Elmore,1,01051,87755,7.727801e+06,57.270124,Relatively Low,2023,33860,01051,"Montgomery, AL",33860,2023,205.86250,195.3500,10.51250
16,Alabama,AL,Etowah,1,01055,103320,2.305295e+07,82.246262,Relatively Low,2023,23460,01055,"Gadsden, AL",23460,2023,290.03250,269.5775,20.45500
18,Alabama,AL,Geneva,1,01061,26621,1.045320e+07,65.542475,Relatively Low,2023,20020,01061,"Dothan, AL",20020,2023,242.38500,221.0125,21.37250


In [158]:
nri.to_csv("../../02_processed_data/nri_hpi_data.csv")

In [159]:
nri[nri["nri_ver"] == 2022]

,state,stateabbrv,county,statefips,stcofips,population,risk_value,risk_score,risk_ratng,nri_ver,cbsa_code,county_fips5,place_name,place_id,yr,index_nsa,index_prev,hpi_change
